In [1]:
# Imports (todos nesta célula)
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score)

# A preparação escolhida nas tarefas 2 e 3 e a régua de avaliação estão em preprocessing.py,
# o mesmo ficheiro que o 02_preprocessing importa (como o preprocessing.py das práticas)
from preprocessing import (ENCODER_FINAL, SCALER_FINAL, TARGET, compare, evaluate, fit_predict,
                           load_train, prepare_selected, sem, time_splits, verdict)

In [2]:
# ===== [4.0] FERRAMENTAS - início =====
# Passo 1 - os dados e as 5 divisões por tempo, exatamente como no 02 (passos 1 a 3): o treino
# ordenado por data, sem as 65 linhas sem data, e cada divisão treina no passado e valida no
# que vem a seguir (W5, secção 3.4)
raw, frame = load_train("../data/train.csv")
splits = time_splits(frame)
test_csv = pd.read_csv("../data/test.csv", index_col="ID")

print(f"treino: {len(frame):,} compras ({len(raw) - len(frame)} sem data ficam de fora);"
      f" teste: {len(test_csv):,} compras")
for fold, (train_idx, val_idx) in enumerate(splits, start=1):
    print(f"divisão {fold}: treina com {len(train_idx):,}, valida com {len(val_idx):,}")

treino: 6,469 compras (65 sem data ficam de fora); teste: 986 compras
divisão 1: treina com 1,079, valida com 1,078
divisão 2: treina com 2,157, valida com 1,078
divisão 3: treina com 3,235, valida com 1,078
divisão 4: treina com 4,313, valida com 1,078
divisão 5: treina com 5,391, valida com 1,078


In [3]:
# Passo 2 - verificação: a preparação importada tem de dar exatamente o resultado do 02
# (passo 62: F1 0.7508, accuracy 0.7163). Se o preprocessing.py mudar sem querer, o notebook
# pára aqui com um erro.
selected_run = evaluate(prepare_selected, frame, splits, scaler=SCALER_FINAL,
                        encoder=ENCODER_FINAL)
assert round(selected_run["f1"].mean(), 4) == 0.7508
assert round(selected_run["accuracy"].mean(), 4) == 0.7163
print(f"referência (regressão logística, preparação final): F1 {selected_run['f1'].mean():.4f}"
      f" +- {sem(selected_run['f1']):.4f}, accuracy {selected_run['accuracy'].mean():.4f}")

referência (regressão logística, preparação final): F1 0.7508 +- 0.0074, accuracy 0.7163


In [4]:
# Passo 3 - a régua dos modelos. Nas aulas, cada definição de um modelo é medida no treino e na
# validação, lado a lado (W6 a W9, "The settings of ..."): um F1 de treino muito acima do de
# validação é sobreajuste; dois valores baixos e próximos são subajuste (W5, "The Complete
# Model Selection Workflow", ponto 2). A tarefa 3 mostrou que também é preciso ver quantas
# vezes o modelo prevê 1 (passos 47 e 59 do 02).
def run_model(model, threshold=None):
    """Em cada divisão: F1 e accuracy na validação, F1 no treino e % de previsões 1.

    `model=None` é a regressão logística da referência; `threshold=None` usa o limiar 0.5.
    """
    rows = []
    for train_idx, val_idx in splits:
        train, valid = frame.iloc[train_idx], frame.iloc[val_idx]
        predicted = fit_predict(prepare_selected, train, valid, SCALER_FINAL, ENCODER_FINAL,
                                model, None, threshold)
        on_train = fit_predict(prepare_selected, train, train, SCALER_FINAL, ENCODER_FINAL,
                               model, None, threshold)
        rows.append({"f1": f1_score(valid[TARGET], predicted),
                     "accuracy": accuracy_score(valid[TARGET], predicted),
                     "treino f1": f1_score(train[TARGET], on_train),
                     "% de 1": 100 * predicted.mean()})
    return pd.DataFrame(rows)


def sweep(models, threshold=None):
    """Cada modelo contra a referência, nas mesmas 5 divisões, com a regra dos 2 x SEM."""
    runs = {name: run_model(model, threshold) for name, model in models.items()}
    table = compare(runs, reference, metric="f1").astype({"f1": float, "accuracy": float,
                                                           "vs base": float, "SEM": float})
    table["2 x SEM"] = 2 * table["SEM"]
    table["veredicto"] = verdict(table)
    table["F1 treino"] = [runs[name]["treino f1"].mean() for name in table.index]
    table["treino - validação"] = table["F1 treino"] - table["f1"]
    table["% de 1"] = [runs[name]["% de 1"].mean() for name in table.index]
    shown = ["f1", "vs base", "2 x SEM", "melhor em", "veredicto", "F1 treino",
             "treino - validação", "accuracy", "% de 1"]
    return table[shown], runs


reference = run_model(None)
assert (reference["f1"] == selected_run["f1"]).all()   # a mesma régua que o evaluate
check, _ = sweep({"regressão logística (referência)": None})
print(check.round(4).to_string())

                                      f1  vs base  2 x SEM melhor em veredicto  F1 treino  treino - validação  accuracy  % de 1
regressão logística (referência)  0.7508      0.0      0.0    0 de 5    empata     0.7316             -0.0192    0.7163  57.885


In [5]:
# ===== [4.0] FERRAMENTAS - fim =====
# Passo 4 - o que o 4.0 entrega aos passos seguintes:
#   - reference: a regressão logística por defeito, F1 0.7508, accuracy 0.7163
#   - sweep({"nome": modelo, ...}): a tabela de cada pacote (4.2 a 4.5) contra a referência
#   - threshold=: o limiar de decisão, para o 4.6 (W5, secção 2.7)
# A referência tem F1 de treino 0.7316, abaixo do de validação (0.7508): não há sinal de
# sobreajuste. Um modelo mais flexível pode aproveitar mais dos dados; é o que o 4.2 a 4.5 testam.
print("4.0 pronto: reference, run_model e sweep")

4.0 pronto: reference, run_model e sweep


In [6]:
# ===== [4.1] ESTRATÉGIA E MÉTRICA - início =====
# Passo 5 - a estratégia de avaliação: 5 divisões por tempo (TimeSeriesSplit, W5, secção 3.4).
# O homework pede uma estratégia e que se explique porque serve este problema:
#   1. o test.csv é posterior ao train.csv (02, passo 2): a pergunta real é "prever o futuro a
#      partir do passado", e cada divisão faz exatamente isso
#   2. 94.5% das compras do teste são de clientes que já aparecem no treino (02, passo 2): o
#      modelo vai prever clientes que conhece, mas sempre em datas posteriores
#   3. o holdout e o K-Fold ao acaso (W5, secções 2 e 3.1) poriam compras do futuro no treino e
#      avaliariam com o passado, o que dá resultados melhores do que os reais
#   - 5 divisões em vez de 1: um só corte é "uma amostra de tamanho um" (W5, key takeaways);
#     com 5 vê-se a média e o erro-padrão, e as diferenças comparam-se divisão a divisão
#   - limitação conhecida: não há intervalo entre treino e validação; o alvo de uma compra de
#     treino pode olhar 90 dias para dentro da validação (02, passo 3)
#   - o test.csv nunca é usado para escolher nada: só para a previsão final (W5, secção 2.1)
print("estratégia: TimeSeriesSplit(n_splits=5), treino sempre anterior à validação")

estratégia: TimeSeriesSplit(n_splits=5), treino sempre anterior à validação


In [7]:
# Passo 6 - as métricas da W5 (secções 2.2 a 2.5) para a referência e para o piso "prever
# sempre 1" (W5, célula 110), divisão a divisão. Precision: das compras que o modelo diz que
# vão ter recompra, quantas têm. Recall: das compras com recompra, quantas o modelo encontra.
metric_rows = []
for fold, (train_idx, val_idx) in enumerate(splits, start=1):
    train, valid = frame.iloc[train_idx], frame.iloc[val_idx]
    truth = valid[TARGET]
    for name, predicted in [("referência", fit_predict(prepare_selected, train, valid,
                                                       SCALER_FINAL, ENCODER_FINAL)),
                            ("prever sempre 1", np.ones(len(valid), dtype=int))]:
        metric_rows.append({"divisão": fold, "previsão": name,
                            "precision": precision_score(truth, predicted),
                            "recall": recall_score(truth, predicted),
                            "f1": f1_score(truth, predicted),
                            "accuracy": accuracy_score(truth, predicted),
                            "% de 1 real": 100 * truth.mean()})
metrics = pd.DataFrame(metric_rows)
shown = ["precision", "recall", "f1", "accuracy"]
print("referência, divisão a divisão:")
print(metrics[metrics["previsão"] == "referência"].set_index("divisão")[shown + ["% de 1 real"]]
      .round(4).to_string(), "\n")
print("média das 5 divisões:")
print(metrics.groupby("previsão")[shown].mean().round(4).to_string())

referência, divisão a divisão:
         precision  recall      f1  accuracy  % de 1 real
divisão                                                  
1           0.7008  0.8365  0.7627    0.6957      58.4416
2           0.7178  0.7639  0.7401    0.7134      53.4323
3           0.7791  0.7327  0.7552    0.7263      57.6067
4           0.7359  0.7201  0.7279    0.7199      52.0408
5           0.7862  0.7512  0.7683    0.7263      60.3896 

média das 5 divisões:
                 precision  recall      f1  accuracy
previsão                                            
prever sempre 1     0.5638  1.0000  0.7206    0.5638
referência          0.7439  0.7609  0.7508    0.7163


In [8]:
# Passo 7 - a matriz de confusão da referência (W5, secção 2.1, Step 26), somando as 5
# validações. O scikit-learn arruma-a assim: [[TN FP] [FN TP]]
#   TN: sem recompra, previsto sem recompra   FP: sem recompra, previsto com (falso alarme)
#   FN: com recompra, previsto sem (falha)    TP: com recompra, previsto com recompra
matrix = sum(confusion_matrix(frame[TARGET].iloc[val_idx],
                              fit_predict(prepare_selected, frame.iloc[train_idx],
                                          frame.iloc[val_idx], SCALER_FINAL, ENCODER_FINAL))
             for train_idx, val_idx in splits)
matrix_table = pd.DataFrame(matrix, index=["real 0 (sem recompra)", "real 1 (com recompra)"],
                            columns=["previsto 0", "previsto 1"])
print(matrix_table.to_string(), "\n")
(tn, fp), (fn, tp) = matrix
print(f"{tn + fp + fn + tp:,} compras de validação; acertos {tn + tp:,}"
      f" ({100 * (tn + tp) / matrix.sum():.1f}%)")
print(f"falsos alarmes (FP): {fp:,} — {100 * fp / (tn + fp):.1f}% das compras sem recompra")
print(f"falhas (FN): {fn:,} — {100 * fn / (fn + tp):.1f}% das compras com recompra")

                       previsto 0  previsto 1
real 0 (sem recompra)        1546         805
real 1 (com recompra)         724        2315 

5,390 compras de validação; acertos 3,861 (71.6%)
falsos alarmes (FP): 805 — 34.2% das compras sem recompra
falhas (FN): 724 — 23.8% das compras com recompra


In [9]:
# ===== [4.1] ESTRATÉGIA E MÉTRICA - fim =====
# Passo 8 - a métrica: F1 (escolha da equipa), com a accuracy sempre ao lado.
#   - o F1 é a média harmónica da precision e da recall da classe 1, a compra com recompra
#     (W5, secção 2.5): só é alto se o modelo encontrar quem volta (recall) sem encher a lista
#     de falsos alarmes (precision). É a classe que interessa à empresa: o enunciado do projeto
#     quer prever a recompra para "more targeted commercial actions"
#   - a referência equilibra as duas: precision 0.744, recall 0.761; os erros dividem-se entre
#     805 falsos alarmes e 724 falhas (passo 7)
#   - o limite do F1, que a própria W5 aponta (célula 111): quando a classe positiva é a
#     maioria, prever sempre 1 já tem F1 alto. Aqui 56% das compras têm recompra, e prever
#     sempre 1 dá F1 0.721; a referência só lhe ganha +0.030 no F1, mas +0.153 na accuracy
#     (0.564 -> 0.716). Por isso a accuracy aparece sempre ao lado: mostra que o modelo
#     aprendeu, coisa que o F1 sozinho quase esconde
#   - a accuracy também serve aqui (as classes estão perto de 56/44, não muito desequilibradas),
#     mas trata da mesma forma acertar quem volta e quem não volta
#   - a confirmar: a métrica da competição no Kaggle ("Overview" -> "Evaluation")
print("métrica: F1 (decide); accuracy, precision e recall ao lado")

métrica: F1 (decide); accuracy, precision e recall ao lado


In [10]:
# ===== [4.2] REGRESSÃO LOGÍSTICA - início =====
# Passo 9 - C (o inverso da força da penalização) para cada forma de penalização, uma definição
# de cada vez, como na W6 (células 45 a 53, Step 10). Os imports ficam aqui para não mexer na
# célula de imports. C=1 com L2 é o modelo por defeito; na W6 todos levam max_iter=5000, e o
# saga (L1 e elastic net) leva também random_state, porque percorre as linhas por ordem aleatória.
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

from preprocessing import RANDOM_STATE

c_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
penalties = {"L2": LogisticRegression(max_iter=5000),
             "L1": LogisticRegression(solver="saga", l1_ratio=1.0, max_iter=5000,
                                      random_state=RANDOM_STATE),
             "elastic net": LogisticRegression(solver="saga", l1_ratio=0.5, max_iter=5000,
                                               random_state=RANDOM_STATE)}
c_table, c_runs = sweep({f"{family}, C={value:g}": clone(model).set_params(C=value)
                         for family, model in penalties.items() for value in c_values})
c_table.round(4)

,f1,vs base,2 x SEM,melhor em,veredicto,F1 treino,treino - validação,accuracy,% de 1
"L2, C=0.001",0.7367,-0.0141,0.0203,1 de 5,empata,0.7169,-0.0198,0.6191,87.9035
"L2, C=0.01",0.7394,-0.0114,0.0105,1 de 5,pior,0.7170,-0.0224,0.6850,65.1020
"L2, C=0.1",0.7485,-0.0023,0.0072,1 de 5,empata,0.7236,-0.0250,0.7087,59.9814
"L2, C=1",0.7508,0.0000,0.0000,0 de 5,empata,0.7316,-0.0192,0.7163,57.8850
"L2, C=10",0.7499,-0.0010,0.0034,2 de 5,empata,0.7310,-0.0188,0.7145,58.1447
"L2, C=100",0.7478,-0.0031,0.0051,2 de 5,empata,0.7304,-0.0174,0.7122,58.2189
"L1, C=0.001",0.7206,-0.0303,0.0113,0 de 5,pior,0.7119,-0.0087,0.5638,100.0000
"L1, C=0.01",0.7126,-0.0383,0.0117,0 de 5,pior,0.7003,-0.0123,0.6284,73.7291
"L1, C=0.1",0.7427,-0.0082,0.0053,1 de 5,pior,0.7153,-0.0274,0.7032,59.3878
"L1, C=1",0.7523,0.0015,0.0022,4 de 5,empata,0.7310,-0.0212,0.7195,57.2356


In [11]:
# Passo 10 - sem penalização (C=np.inf) contra a penalização por defeito (C=1), como na W6
# (célula 56, Step 11). Na W6 os rótulos quase não mudaram: com C=1 a penalização já é leve.
none_table, none_runs = sweep({"L2, C=1": LogisticRegression(C=1.0, max_iter=5000),
                               "sem penalização, C=inf": LogisticRegression(C=np.inf,
                                                                            max_iter=5000)})
none_table.round(4)

,f1,vs base,2 x SEM,melhor em,veredicto,F1 treino,treino - validação,accuracy,% de 1
"L2, C=1",0.7508,0.0000,0.0000,0 de 5,empata,0.7316,-0.0192,0.7163,57.8850
"sem penalização, C=inf",0.7456,-0.0053,0.0065,2 de 5,empata,0.7313,-0.0142,0.7096,58.2189


In [12]:
# Passo 11 - class_weight (W6, célula 61, Step 12). "balanced" dá a cada classe um peso inverso
# à sua frequência. Aqui a classe 1 é a maioria (56%), por isso "balanced" tira peso à classe em
# que o F1 é calculado: espera-se menos previsões 1 (ver "% de 1"), menos recall e menos F1.
weight_table, weight_runs = sweep({
    f"class_weight={weight}": LogisticRegression(class_weight=weight, max_iter=5000)
    for weight in [None, "balanced"]})
weight_table.round(4)

,f1,vs base,2 x SEM,melhor em,veredicto,F1 treino,treino - validação,accuracy,% de 1
class_weight=None,0.7508,0.0000,0.0000,0 de 5,empata,0.7316,-0.0192,0.7163,57.8850
class_weight=balanced,0.7382,-0.0126,0.0089,1 de 5,pior,0.7153,-0.0229,0.7178,52.0965


In [13]:
# Passo 12 - fit_intercept (W6, célula 66, Step 13): sem a constante, a compra com todas as
# colunas a zero fica forçada a probabilidade 0.5 e os outros coeficientes compensam.
intercept_table, intercept_runs = sweep({
    f"fit_intercept={value}": LogisticRegression(fit_intercept=value, max_iter=5000)
    for value in [True, False]})
intercept_table.round(4)

,f1,vs base,2 x SEM,melhor em,veredicto,F1 treino,treino - validação,accuracy,% de 1
fit_intercept=True,0.7508,0.0000,0.0000,0 de 5,empata,0.7316,-0.0192,0.7163,57.8850
fit_intercept=False,0.7515,0.0007,0.0023,3 de 5,empata,0.7316,-0.0199,0.7167,58.0334


In [14]:
# Passo 13 - a grelha (W6, célula 71, Step 14), só com os valores que não pioraram nos passos
# 9 a 12 (veredicto "empata" ou "melhor" contra a referência), como pede o plano da equipa: a
# grelha completa da W6 (76 combinações) demoraria mais de 10 minutos, porque o saga é lento.
# Como na W6, C=np.inf é uma sub-grelha à parte: sem penalização não há força nem forma a escolher.
def kept(table, values, label):
    """Os valores de `values` cuja linha `label(value)` da tabela não piorou."""
    return [value for value in values if table.loc[label(value), "veredicto"] != "pior"]


grid_c = {family: kept(c_table, c_values, lambda value, f=family: f"{f}, C={value:g}")
          for family in penalties}
grid_weight = kept(weight_table, [None, "balanced"], lambda value: f"class_weight={value}")
grid_intercept = kept(intercept_table, [True, False], lambda value: f"fit_intercept={value}")
grid_none = none_table.loc["sem penalização, C=inf", "veredicto"] != "pior"

grid = {}
for family, model in penalties.items():
    for value in grid_c[family]:
        for weight in grid_weight:
            for intercept in grid_intercept:
                grid[f"{family}, C={value:g}, class_weight={weight}, fit_intercept={intercept}"] = \
                    clone(model).set_params(C=value, class_weight=weight, fit_intercept=intercept)
if grid_none:
    for weight in grid_weight:
        for intercept in grid_intercept:
            grid[f"sem penalização, class_weight={weight}, fit_intercept={intercept}"] = \
                LogisticRegression(C=np.inf, class_weight=weight, fit_intercept=intercept,
                                   max_iter=5000)
print("valores que ficam na grelha:", {"C": grid_c, "class_weight": grid_weight,
                                       "fit_intercept": grid_intercept, "C=inf": grid_none})
print(f"{len(grid)} combinações (a grelha completa da W6 tem 76)\n")

grid_table, grid_runs = sweep(grid)
print("as 5 melhores pelo F1 médio, e o modelo por defeito:")
default_name = "L2, C=1, class_weight=None, fit_intercept=True"
top = grid_table.sort_values("f1", ascending=False).head(5)
if default_name in grid_table.index and default_name not in top.index:
    top = pd.concat([top, grid_table.loc[[default_name]]])
top.round(4)

valores que ficam na grelha: {'C': {'L2': [0.001, 0.1, 1.0, 10.0, 100.0], 'L1': [1.0, 10.0, 100.0], 'elastic net': [1.0, 10.0, 100.0]}, 'class_weight': [None], 'fit_intercept': [True, False], 'C=inf': True}
24 combinações (a grelha completa da W6 tem 76)

as 5 melhores pelo F1 médio, e o modelo por defeito:


,f1,vs base,2 x SEM,melhor em,veredicto,F1 treino,treino - validação,accuracy,% de 1
"L1, C=100, class_weight=None, fit_intercept=False",0.7540,0.0031,0.0028,5 de 5,melhor,0.7309,-0.0231,0.7219,57.1429
"L1, C=100, class_weight=None, fit_intercept=True",0.7538,0.0029,0.0025,5 de 5,melhor,0.7309,-0.0228,0.7217,57.1243
"L1, C=10, class_weight=None, fit_intercept=False",0.7535,0.0027,0.0019,5 de 5,melhor,0.7304,-0.0231,0.7215,57.0315
"L1, C=10, class_weight=None, fit_intercept=True",0.7535,0.0027,0.0019,5 de 5,melhor,0.7304,-0.0231,0.7215,57.0315
"elastic net, C=10, class_weight=None, fit_intercept=True",0.7524,0.0016,0.0018,4 de 5,empata,0.7310,-0.0214,0.7199,57.2356
"L2, C=1, class_weight=None, fit_intercept=True",0.7508,0.0000,0.0000,0 de 5,empata,0.7316,-0.0192,0.7163,57.8850


In [15]:
# ===== [4.2] REGRESSÃO LOGÍSTICA - fim =====
# Passo 14 - o melhor modelo logístico. O que os passos 9 a 13 mostraram (contra a referência,
# F1 0.7508, accuracy 0.7163):
#   - C: penalizações fortes (C <= 0.1) pioram ou ficam perto de prever sempre 1; a partir de
#     C=1 as três formas ficam perto umas das outras, como na W6. Só o L1 com C=10 e C=100
#     passa os 2 x SEM (+0.0027 e +0.0029, melhor em 5 de 5); o L2 C=0.001 entrou na grelha
#     só porque o seu erro-padrão é grande (prevê 1 em 88% das compras)
#   - sem penalização, fit_intercept=False: empatam (como na W6)
#   - class_weight="balanced": pior (-0.0126); prevê menos 1s (52% contra 58%) e baixa a recall,
#     como a W6 previa quando a classe 1 é a maioria
#   - nenhuma definição tem sobreajuste: o F1 de treino fica sempre abaixo do de validação
# A grelha (passo 13) coloca no topo 4 combinações de L1 com C=10 ou 100, que empatam entre si.
# Num empate fica a mais simples, a que muda menos coisas ao modelo por defeito: L1, C=10, com
# constante (muda a forma da penalização e o C). Cuidado: foi escolhida entre 24 combinações nas
# mesmas divisões que a medem, por isso o ganho é otimista (W6, "Is the grid worth running?").
best_name = grid_table["f1"].idxmax()
chosen_name = "L1, C=10, class_weight=None, fit_intercept=True"
for metric in ["f1", "accuracy"]:
    tie = compare({f"{chosen_name} vs o topo da grelha": grid_runs[chosen_name]},
                  grid_runs[best_name], metric=metric)
    tie["2 x SEM"] = 2 * tie["SEM"]
    print(f"[{metric}]", tie[["vs base", "2 x SEM", "melhor em"]].round(4).to_string(header=metric == "f1"))

BEST_LOGISTIC = clone(grid[chosen_name])   # para o 4.6 e o 4.7
logistic_runs = grid_runs[chosen_name]
print(f"\nmelhor regressão logística: {chosen_name}")
print(grid_table.loc[[chosen_name]].round(4).to_string())

[f1]                                                                       vs base   2 x SEM melhor em
L1, C=10, class_weight=None, fit_intercept=True vs o topo da grelha -0.000474  0.001257    2 de 5
[accuracy] L1, C=10, class_weight=None, fit_intercept=True vs o topo da grelha -0.000371  0.000946  1 de 5

melhor regressão logística: L1, C=10, class_weight=None, fit_intercept=True
                                                     f1  vs base  2 x SEM melhor em veredicto  F1 treino  treino - validação  accuracy   % de 1
L1, C=10, class_weight=None, fit_intercept=True  0.7535   0.0027   0.0019    5 de 5    melhor     0.7304             -0.0231    0.7215  57.0315


In [16]:
# ===== [4.3] NAIVE BAYES - início =====
# Passo 15 - o Naive Bayes gaussiano por defeito (W7, célula 35), contra a referência. Supõe
# que, dentro de cada classe, as colunas são independentes e normais; aqui não são (há colunas
# one-hot de 0/1 e contagens correlacionadas), por isso pode ficar abaixo da regressão logística,
# como na W7 (0.78 contra 0.85). Não aprende nada iterativamente: só conta, faz médias e
# variâncias por classe, por isso é muito rápido.
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import GaussianNB

nb_default_table, nb_default_runs = sweep({"GaussianNB()": GaussianNB()})
nb_default_table.round(4)

,f1,vs base,2 x SEM,melhor em,veredicto,F1 treino,treino - validação,accuracy,% de 1
GaussianNB(),0.5382,-0.2126,0.1509,0 de 5,pior,0.416,-0.1222,0.5878,43.9889


In [17]:
# Passo 16 - var_smoothing (W7, Step 21): soma a todas as variâncias uma fração da maior
# variância das colunas, para nenhuma ser zero. Previsão da W7: nada muda enquanto essa
# constante for pequena; quando chega ao tamanho das variâncias, o modelo cai para o prior.
smoothing_values = [1e-12, 1e-9, 1e-6, 1e-3, 1e-1, 1.0, 10.0]
smoothing_table, smoothing_runs = sweep({f"var_smoothing={value:g}": GaussianNB(var_smoothing=value)
                                         for value in smoothing_values})
smoothing_table.round(4)

,f1,vs base,2 x SEM,melhor em,veredicto,F1 treino,treino - validação,accuracy,% de 1
var_smoothing=1e-12,0.4968,-0.2540,0.1825,0 de 5,pior,0.3805,-0.1164,0.5623,43.8590
var_smoothing=1e-09,0.5382,-0.2126,0.1509,0 de 5,pior,0.4160,-0.1222,0.5878,43.9889
var_smoothing=1e-06,0.6108,-0.1401,0.1177,0 de 5,pior,0.4997,-0.1111,0.6423,42.8015
var_smoothing=0.001,0.6747,-0.0762,0.0320,0 de 5,pior,0.6343,-0.0403,0.6772,43.7662
var_smoothing=0.1,0.6856,-0.0652,0.0217,0 de 5,pior,0.6661,-0.0195,0.6755,47.6067
var_smoothing=1,0.7077,-0.0431,0.0186,0 de 5,pior,0.6856,-0.0222,0.6787,54.2301
var_smoothing=10,0.7281,-0.0227,0.0186,1 de 5,pior,0.7112,-0.0169,0.5878,95.1577


In [18]:
# Passo 17 - priors (W7, Step 22): a crença sobre cada classe antes de ver as colunas, na ordem
# [classe 0, classe 1]. None usa a frequência do treino (cerca de 44% / 56%). Um prior maior na
# classe 1 faz o modelo prever mais 1s, o que sobe a recall e, aqui, o F1, sem acertar mais:
# por isso a accuracy e a "% de 1" ficam ao lado.
prior_values = [None, [0.8, 0.2], [0.6, 0.4], [0.5, 0.5], [0.2, 0.8]]
prior_table, prior_runs = sweep({f"priors={prior}": GaussianNB(priors=prior)
                                 for prior in prior_values})
prior_table.round(4)

,f1,vs base,2 x SEM,melhor em,veredicto,F1 treino,treino - validação,accuracy,% de 1
priors=None,0.5382,-0.2126,0.1509,0 de 5,pior,0.4160,-0.1222,0.5878,43.9889
"priors=[0.8, 0.2]",0.5212,-0.2296,0.1748,0 de 5,pior,0.4021,-0.1191,0.5909,40.7792
"priors=[0.6, 0.4]",0.5310,-0.2199,0.1595,0 de 5,pior,0.4112,-0.1198,0.5878,42.8757
"priors=[0.5, 0.5]",0.5378,-0.2130,0.1527,0 de 5,pior,0.4155,-0.1223,0.5889,43.7291
"priors=[0.2, 0.8]",0.5556,-0.1952,0.1342,0 de 5,pior,0.4300,-0.1256,0.5907,46.0668


In [19]:
# Passo 18 - calibração (W7, Step 23): o CalibratedClassifierCV treina o Naive Bayes em 5
# partes do treino de cada divisão e aprende, nas partes que não viu, a corrigir as suas
# probabilidades. Na W7 a calibração isotónica subiu o F1 de 0.7765 para 0.7977. As partes
# internas saem só do treino de cada divisão, por isso a validação nunca entra.
def naive_bayes(var_smoothing=1e-9, priors=None, method=None):
    """GaussianNB com estas definições; com `method`, calibrado como na W7."""
    model = GaussianNB(var_smoothing=var_smoothing, priors=priors)
    if method is None:
        return model
    return CalibratedClassifierCV(model, method=method)


calibration_values = [None, "sigmoid", "isotonic"]
calibration_table, calibration_runs = sweep({f"calibração={method}": naive_bayes(method=method)
                                             for method in calibration_values})
calibration_table.round(4)

,f1,vs base,2 x SEM,melhor em,veredicto,F1 treino,treino - validação,accuracy,% de 1
calibração=None,0.5382,-0.2126,0.1509,0 de 5,pior,0.4160,-0.1222,0.5878,43.9889
calibração=sigmoid,0.7105,-0.0404,0.0293,0 de 5,pior,0.6978,-0.0126,0.6104,80.0557
calibração=isotonic,0.7178,-0.0331,0.0100,0 de 5,pior,0.7093,-0.0084,0.6855,55.5102


In [20]:
# Passo 19 - a grelha completa da W7 (Step 24): 7 var_smoothing x 5 priors x 3 calibrações = 105
# combinações. O Naive Bayes é rápido, por isso não é preciso cortar; e a W7 mostra que a
# calibração e o var_smoothing só ajudam em conjunto. Ao lado, o veredicto na accuracy: um
# prior que só faz prever mais 1s sobe o F1 sem acertar mais (passo 6, tarefa 3).
nb_grid = {f"var_smoothing={value:g}, priors={prior}, calibração={method}":
           naive_bayes(value, prior, method)
           for value in smoothing_values for prior in prior_values for method in calibration_values}
nb_grid_table, nb_grid_runs = sweep(nb_grid)
accuracy_check = compare(nb_grid_runs, reference, metric="accuracy").astype(
    {"accuracy": float, "vs base": float, "SEM": float})
nb_grid_table["accuracy vs base"] = accuracy_check["vs base"]
nb_grid_table["veredicto accuracy"] = verdict(accuracy_check)

print(f"{len(nb_grid)} combinações; as 5 melhores pelo F1 médio, e o Naive Bayes por defeito:")
nb_default_name = "var_smoothing=1e-09, priors=None, calibração=None"
nb_top = nb_grid_table.sort_values("f1", ascending=False).head(5)
nb_top = pd.concat([nb_top, nb_grid_table.loc[[nb_default_name]]])
nb_top.round(4)

105 combinações; as 5 melhores pelo F1 médio, e o Naive Bayes por defeito:


,f1,vs base,2 x SEM,melhor em,veredicto,F1 treino,treino - validação,accuracy,% de 1,accuracy vs base,veredicto accuracy
"var_smoothing=1, priors=[0.2, 0.8], calibração=None",0.7390,-0.0119,0.0163,1 de 5,empata,0.7175,-0.0215,0.6212,88.9981,-0.0952,pior
"var_smoothing=1e-09, priors=[0.8, 0.2], calibração=sigmoid",0.7351,-0.0157,0.0180,2 de 5,empata,0.7199,-0.0152,0.6239,85.1206,-0.0924,pior
"var_smoothing=1, priors=[0.2, 0.8], calibração=sigmoid",0.7349,-0.0160,0.0103,1 de 5,pior,0.7199,-0.0150,0.6688,68.8683,-0.0475,pior
"var_smoothing=1e-09, priors=[0.6, 0.4], calibração=sigmoid",0.7335,-0.0173,0.0168,1 de 5,pior,0.7205,-0.0130,0.6184,86.4193,-0.0980,pior
"var_smoothing=1, priors=[0.5, 0.5], calibração=isotonic",0.7321,-0.0187,0.0061,0 de 5,pior,0.7137,-0.0184,0.6820,62.7644,-0.0343,pior
"var_smoothing=1e-09, priors=None, calibração=None",0.5382,-0.2126,0.1509,0 de 5,pior,0.4160,-0.1222,0.5878,43.9889,-0.1286,pior


In [21]:
# ===== [4.3] NAIVE BAYES - fim =====
# Passo 20 - o melhor Naive Bayes. O que os passos 15 a 19 mostraram:
#   - por defeito é muito pior do que a regressão logística: F1 0.538, accuracy 0.588, e o F1
#     de treino (0.42) ainda mais baixo: subajuste, porque as suposições do modelo falham aqui
#   - var_smoothing maior ajuda até 1; com 10 o modelo cai para "prever quase sempre 1" (95%)
#   - priors sozinhos quase não mexem; a calibração é o que mais ajuda (isotónica: F1 0.718),
#     como na W7
#   - na grelha, as combinações com o F1 mais alto preveem 1 em 85 a 89% das compras e a sua
#     accuracy cai para 0.62: ganham o F1 da mesma forma que "prever sempre 1" (passo 6)
# A regra, para não premiar esse truque:
#   1. ficam de fora as combinações com accuracy pior (acima do ruído) do que a mais certeira
#      da grelha
#   2. das restantes, a de maior F1
#   3. num empate, a que muda menos definições ao Naive Bayes por defeito
most_accurate = nb_grid_table["accuracy"].idxmax()
accuracy_vs_best = compare(nb_grid_runs, nb_grid_runs[most_accurate], metric="accuracy").astype(
    {"vs base": float, "SEM": float})
allowed = nb_grid_table[verdict(accuracy_vs_best) != "pior"]
top_name = allowed["f1"].idxmax()
nb_chosen_name = "var_smoothing=1, priors=None, calibração=isotonic"
print(f"mais certeira: {most_accurate} (accuracy {nb_grid_table.loc[most_accurate, 'accuracy']:.4f})")
print(f"{len(allowed)} de {len(nb_grid_table)} combinações sem accuracy pior; a de maior F1:"
      f" {top_name}\n")
for metric in ["f1", "accuracy"]:
    tie = compare({f"{nb_chosen_name} vs {top_name}": nb_grid_runs[nb_chosen_name]},
                  nb_grid_runs[top_name], metric=metric)
    tie["2 x SEM"] = 2 * tie["SEM"]
    print(f"[{metric}]", tie[["vs base", "2 x SEM", "melhor em"]].round(4)
          .to_string(header=metric == "f1"))
# Empatam: fica a isotónica com priors=None, que muda duas definições (var_smoothing e a
# calibração) em vez de três, e prevê 1 menos vezes (63% contra 69%). Com a calibração isotónica
# os 5 priors dão o mesmo resultado (passo 19), como a W7 explica: o prior não muda a ordem das
# compras, e a isotónica só usa essa ordem.
#   - continua pior do que a regressão logística (F1 -0.019, accuracy -0.034): entra na tabela
#     do 4.7 como um algoritmo diferente, não como candidato a modelo final
#   - escolhido entre 105 combinações nas mesmas divisões que o medem: o número é otimista
BEST_NAIVE_BAYES = clone(nb_grid[nb_chosen_name])   # para o 4.6 e o 4.7
naive_bayes_runs = nb_grid_runs[nb_chosen_name]
print(f"\nmelhor Naive Bayes: {nb_chosen_name}")
print(nb_grid_table.loc[[nb_chosen_name]].round(4).to_string())

mais certeira: var_smoothing=0.001, priors=[0.5, 0.5], calibração=sigmoid (accuracy 0.6892)
60 de 105 combinações sem accuracy pior; a de maior F1: var_smoothing=1, priors=[0.2, 0.8], calibração=sigmoid

[f1]                                                                                                               vs base   2 x SEM melhor em
var_smoothing=1, priors=None, calibração=isotonic vs var_smoothing=1, priors=[0.2, 0.8], calibração=sigmoid -0.002715  0.007403    2 de 5
[accuracy] var_smoothing=1, priors=None, calibração=isotonic vs var_smoothing=1, priors=[0.2, 0.8], calibração=sigmoid  0.013173  0.013913  3 de 5

melhor Naive Bayes: var_smoothing=1, priors=None, calibração=isotonic
                                                       f1  vs base  2 x SEM melhor em veredicto  F1 treino  treino - validação  accuracy   % de 1  accuracy vs base veredicto accuracy
var_smoothing=1, priors=None, calibração=isotonic  0.7321  -0.0187   0.0061    0 de 5      pior     0.7137        

In [10]:
# ===== SUBMISSÃO AO KAGGLE - início =====
# Passo 21 - a célula que gera o ficheiro passou do 02 para aqui: o modelo escolhe-se neste
# notebook, e o enunciado do projeto (V. Deployment) pede que a última parte aplique o pipeline
# ao test.csv. Primeiro, a prova de que nada mudou na passagem: sem limiar, a preparação final
# e a regressão logística, ajustadas no treino inteiro, dão exatamente a Version04.
version_04 = pd.read_csv("../submissions/ML_Group_37_Version04.csv", index_col="ID")[TARGET]
same_as_04 = fit_predict(prepare_selected, frame, test_csv, SCALER_FINAL, ENCODER_FINAL)
assert (same_as_04 == version_04.loc[test_csv.index].to_numpy()).all()
print("sem limiar, o 03 reproduz a Version04: 986 de 986 previsões iguais")

sem limiar, o 03 reproduz a Version04: 986 de 986 previsões iguais


In [11]:
# Passo 22 - a submissão. Histórico (score = a parte visível da leaderboard, 70% do teste):
#   Version01  preparação do 2.2, regressão logística                 F1 local 0.734  Kaggle 0.77
#   Version02  as mesmas previsões da Version01                                        Kaggle ?
#   Version03  preparação final do 2.7                                F1 local 0.748  Kaggle ?
#   Version04  + as features da tarefa 3 (sem as 14 colunas)          F1 local 0.751  Kaggle ?
#   Version05  a Version04 com limiar 0.45 (W5, secção 2.7)            F1 local 0.753  Kaggle ?
# A Version05 é uma sonda, não uma melhoria provada: localmente o limiar 0.45 empata no F1
# (+0.0026, 2 x SEM = 0.0043) e piora a accuracy (-0.011). Prevê mais 1s (697 contra 649),
# como o sample_submission (741), que teve 0.779 contra os 0.77 da Version01. Se subir, é sinal
# de que a métrica do Kaggle premeia prever mais 1s. Não se escolhe o modelo final por isto.
# VERSION e THRESHOLD são escritos à mão: correr o notebook outra vez volta a gerar o mesmo
# ficheiro, e o notebook entregue tem de reproduzir a submissão escolhida no Kaggle.
TEAM = "ML_Group_37"
VERSION = 5
THRESHOLD = 0.45   # None = o predict do modelo (limiar 0.5)

predictions = fit_predict(prepare_selected, frame, test_csv, SCALER_FINAL, ENCODER_FINAL,
                          threshold=THRESHOLD)
submission = pd.DataFrame({"ID": test_csv.index, TARGET: predictions})

path = f"../submissions/{TEAM}_Version{VERSION:02d}.csv"
submission.to_csv(path, index=False)   # sem o índice do pandas, como pede o guia do Kaggle

sample = pd.read_csv("../data/sample_submission.csv")
print(f"gravado em {path}")
print(f"linhas: {len(submission)}, IDs únicos: {submission['ID'].nunique()},"
      f" mesmos IDs do sample_submission: {submission['ID'].isin(sample['ID']).all()}")
print("previsões:", submission[TARGET].value_counts().sort_index().to_dict())
print("diferentes da Version04:", int((predictions != same_as_04).sum()), "de", len(predictions))

gravado em ../submissions/ML_Group_37_Version05.csv
linhas: 986, IDs únicos: 986, mesmos IDs do sample_submission: True
previsões: {0: 289, 1: 697}
diferentes da Version04: 48 de 986
